In [81]:
import numpy as np
import pandas as pd

df = pd.read_csv('diabetes.csv')
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
515,3,163,70,18,105,31.6,0.268,28,1
118,4,97,60,23,0,28.2,0.443,22,0
722,1,149,68,29,127,29.3,0.349,42,1
662,8,167,106,46,231,37.6,0.165,43,1
468,8,120,0,0,0,30.0,0.183,38,1


In [83]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [85]:
x = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [87]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [89]:
x = scaler.fit_transform(x)

In [91]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,
                                                   random_state=1)

In [139]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [95]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])



/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [97]:
model.fit(x_train, y_train, batch_size=32, epochs=100,
          validation_data=(x_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3616 - loss: 0.7587 - val_accuracy: 0.4610 - val_loss: 0.7162
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5391 - loss: 0.6986 - val_accuracy: 0.6299 - val_loss: 0.6646
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6450 - loss: 0.6541 - val_accuracy: 0.7143 - val_loss: 0.6250
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6792 - loss: 0.6197 - val_accuracy: 0.7662 - val_loss: 0.5945
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7166 - loss: 0.5918 - val_accuracy: 0.7792 - val_loss: 0.5687
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7199 - loss: 0.5676 - val_accuracy: 0.7727 - val_loss: 0.5485
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7378 - loss: 0.5477 - val_accuracy: 0.7987 - val_loss: 0.5311
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7427 - loss: 0.5314 - val_accuracy: 0.7727 - v

# Hyperparameter Tuning
1. How to select appropriate Optimizer.
2. No. of nodes in a layer
3. How to select no. of layers
4. All in one model

In [22]:
import kerastuner as kt

/var/folders/y2/sdgjnlgd3znfjymq07bh0d_00000gn/T/ipykernel_33487/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [24]:
# how to select the appropriate optimizer

def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


    return model




In [26]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
tuner.search(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

Trial 4 Complete [00h 00m 01s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 04s


In [34]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [36]:
model = tuner.get_best_models(num_models=1)[0]

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [38]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch=6, 
          validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7590 - loss: 0.4828 - val_accuracy: 0.7922 - val_loss: 0.4818
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7606 - loss: 0.4761 - val_accuracy: 0.7922 - val_loss: 0.4784
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7638 - loss: 0.4726 - val_accuracy: 0.7987 - val_loss: 0.4750
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7638 - loss: 0.4693 - val_accuracy: 0.7857 - val_loss: 0.4762
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.4659 - val_accuracy: 0.7922 - val_loss: 0.4751
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.4633 - val_accuracy: 0.7922 - val_loss: 0.4726
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.4606 - val_accuracy: 0.7987 - val_loss: 0.4713
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7720 - loss: 0.4585 - val_accuracy: 0.798

In [99]:
# No. of correct nodes in a layer

def build_model2(hp):
    model = Sequential()
    units = hp.Int('units', 8, 128, step=8)

    model.add(Dense(units=units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [101]:
tuner = kt.RandomSearch(build_model2, objective='val_accuracy', max_trials=5)

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [103]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 04s


In [105]:
tuner.get_best_hyperparameters()[0].values

{'units': 96}

# All in One

In [161]:
def build_model3(hp):

    model=Sequential()

    counter = 0
    
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        
        if counter == 0:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8), 
                           activation= hp.Choice('activation'+str(i), values=['relu', 'tanh', 'sigmoid']),
                    input_dim=8
                )
            )
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6])))

        else:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8), 
                           activation= hp.Choice('activation'+str(i), values=['relu', 'tanh', 'sigmoid'])
                )
            )
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6])))

        counter += 1


    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])


    return model


In [163]:
tuner = kt.RandomSearch(build_model3,
                       objective = 'val_accuracy',
                       directory="mydir",
                       project_name='final')

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [165]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Trial 10 Complete [00h 00m 02s]
val_accuracy: 0.7662337422370911

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 19s


In [167]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 5,
 'units0': 40,
 'activation0': 'tanh',
 'dropout0': 0.6,
 'optimizer': 'rmsprop',
 'units1': 64,
 'activation1': 'relu',
 'dropout1': 0.3,
 'units2': 88,
 'activation2': 'tanh',
 'dropout2': 0.3,
 'units3': 112,
 'activation3': 'sigmoid',
 'dropout3': 0.1,
 'units4': 40,
 'activation4': 'relu',
 'dropout4': 0.1,
 'units5': 16,
 'activation5': 'tanh',
 'dropout5': 0.6}

In [169]:
model = tuner.get_best_models(num_models=1)[0]

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [171]:
model.fit(x_train, y_train, epochs=200, initial_epoch=5, validation_data=(x_test, y_test))



Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7394 - loss: 0.5213 - val_accuracy: 0.7792 - val_loss: 0.4653
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7622 - loss: 0.5090 - val_accuracy: 0.7987 - val_loss: 0.4712
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7394 - loss: 0.5218 - val_accuracy: 0.7662 - val_loss: 0.4676
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7508 - loss: 0.5147 - val_accuracy: 0.7987 - val_loss: 0.4596
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7590 - loss: 0.5075 - val_accuracy: 0.7662 - val_loss: 0.4758
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7492 - loss: 0.5226 - val_accuracy: 0.8052 - val_loss: 0.4624
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7459 - loss: 0.4902 - val_accuracy: 0.8117 - val_loss: 0.4638
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7622 - loss: 0.5028 - val_accuracy: 0.7922